# sweep-config-dict — worked example 2: Sweep Config — grid search with fixed and discrete params

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sweep-config-dict`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Grid search exhaustively tries every combination of the listed parameter values. Each parameter in a grid sweep must use either `'values': [...]` (a discrete list) or `'value': x` (a single fixed value). Continuous distributions like `log_uniform_values` are not valid under grid search because there is no finite grid to iterate over.

## Worked solution

**Step 1 — Set method to grid.**
Grid search requires `'method': 'grid'`.

**Step 2 — Metric block.**
Even for grid search you specify a metric so wandb can mark the best run. Here we minimize `'train_loss'`.

**Step 3 — Parameters must be discrete lists or fixed values.**
For `batch_size` we provide a list of candidate sizes. For `optimizer` we want to compare two specific choices. For `weight_decay` we fixed it at `1e-4` (single value). The combination count is 4 × 2 × 1 = 8 runs.

**Step 4 — Verify the structure.**
Each parameter spec has exactly one of `'value'`, `'values'`, or `'distribution'`. Under grid, `'distribution'` entries would cause a schema error in wandb — keep everything as `'value'` or `'values'`.

In [ ]:
def build_grid_sweep_config() -> dict:
    return {
        'method': 'grid',
        'metric': {
            'name': 'train_loss',
            'goal': 'minimize',
        },
        'parameters': {
            'batch_size': {
                'values': [16, 32, 64, 128],
            },
            'optimizer': {
                'values': ['sgd', 'adamw'],
            },
            'weight_decay': {
                'value': 1e-4,
            },
        },
    }

# Demonstrate
cfg = build_grid_sweep_config()
print('method:', cfg['method'])          # grid
print('metric goal:', cfg['metric']['goal'])  # minimize
n_combinations = 1
for spec in cfg['parameters'].values():
    n_combinations *= len(spec.get('values', [spec.get('value')]))
print('total grid combinations:', n_combinations)  # 8